In [1]:
# task 2

In [2]:
#load the data we already saved
import pandas as pd

df = pd.read_csv("books_data.csv")
print(df.shape)
df.head()

(100, 6)


,Title,Price (GBP),Price (INR),Rating,Stock Count,Category
0,A Light in the Attic,51.77,5435.85,3,22,Poetry
1,Tipping the Velvet,53.74,5642.70,1,20,Historical Fiction
2,Soumission,50.10,5260.50,1,20,Fiction
3,Sharp Objects,47.82,5021.10,4,20,Mystery
4,Sapiens: A Brief History of Humankind,54.23,5694.15,5,20,History


In [3]:
#What's the price distribution
df["Price (INR)"].describe()

count     100.000000
mean     3628.873500
std      1537.045757
min      1066.800000
25%      2089.237500
50%      3651.375000
75%      5036.587500
max      6101.550000
Name: Price (INR), dtype: float64

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Title        100 non-null    object 
 1   Price (GBP)  100 non-null    float64
 2   Price (INR)  100 non-null    float64
 3   Rating       100 non-null    int64  
 4   Stock Count  100 non-null    int64  
 5   Category     100 non-null    object 
dtypes: float64(2), int64(2), object(2)
memory usage: 4.8+ KB


In [5]:
#Does price vary by category
df.groupby("Category")["Price (INR)"].mean().sort_values(ascending=False)

Category
Historical Fiction    5642.700000
Politics              5389.650000
Childrens             5168.100000
Health                5150.250000
Self Help             4866.750000
Travel                4742.850000
New Adult             4732.350000
Art                   4638.900000
Fiction               4606.560000
Music                 4535.650000
Science               4510.800000
Mystery               4338.250000
Horror                4121.250000
Philosophy            3887.625000
Science Fiction       3846.150000
Poetry                3805.050000
Food and Drink        3663.030000
History               3563.700000
Business              3500.700000
Nonfiction            3410.662500
Sequential Art        3350.175000
Contemporary          3335.850000
Add a comment         3185.910000
Romance               3139.500000
Thriller              3110.800000
Fantasy               2980.425000
Default               2819.016667
Young Adult           2629.987500
Spirituality          2619.750000
Name:

In [6]:
#how many books per category (important context)
df["Category"].value_counts()

Category
Sequential Art        14
Nonfiction            12
Default                9
Poetry                 7
Add a comment          5
Food and Drink         5
Fiction                5
History                4
Young Adult            4
Fantasy                4
Mystery                3
Music                  3
Thriller               3
Childrens              3
Science Fiction        2
Romance                2
Spirituality           2
Philosophy             2
Contemporary           1
Horror                 1
Health                 1
Science                1
Business               1
New Adult              1
Art                    1
Historical Fiction     1
Travel                 1
Politics               1
Self Help              1
Name: count, dtype: int64

In [7]:
#let's look at which books got this wrong "category":
df[df["Category"] == "Add a comment"]

,Title,Price (GBP),Price (INR),Rating,Stock Count,Category
55,The Torch Is Passed: A Harding Family Story,19.09,2004.45,1,16,Add a comment
63,The Mindfulness and Acceptance Workbook for An...,23.89,2508.45,4,16,Add a comment
70,The Art Forger,40.76,4279.80,3,16,Add a comment
88,On a Midnight Clear,14.07,1477.35,3,16,Add a comment
97,Judo: Seven Steps to Black Belt (an Introducto...,53.90,5659.50,2,16,Add a comment


In [12]:
import requests
from bs4 import BeautifulSoup

# Books.toscrape has a search-like structure via its sitemap; easier: get link from homepage/catalogue directly
# Let's fetch page 3 (where "The Art Forger" appeared, row index 70 in our data ~ page 4 since 20/page)

page_url = "https://books.toscrape.com/catalogue/page-4.html"
response = requests.get(page_url)
response.encoding = 'utf-8'
soup_page = BeautifulSoup(response.text, "html.parser")

books_on_page = soup_page.find_all("article", class_="product_pod")

for book in books_on_page:
    title = book.h3.a["title"]
    if "Art Forger" in title:
        print("Found:", title)
        print("Link:", book.h3.a["href"])

Found: The Art Forger
Link: the-art-forger_930/index.html


In [13]:
from urllib.parse import urljoin

correct_link = urljoin(page_url, "the-art-forger_930/index.html")
response2 = requests.get(correct_link)
response2.encoding = 'utf-8'
soup_correct = BeautifulSoup(response2.text, "html.parser")

breadcrumb_correct = soup_correct.find("ul", class_="breadcrumb")
print(breadcrumb_correct.prettify())

<ul class="breadcrumb">
 <li>
  <a href="../../index.html">
   Home
  </a>
 </li>
 <li>
  <a href="../category/books_1/index.html">
   Books
  </a>
 </li>
 <li>
  <a href="../category/books/add-a-comment_18/index.html">
   Add a comment
  </a>
 </li>
 <li class="active">
  The Art Forger
 </li>
</ul>



In [14]:
#relabel this specific known site quirk:
# "Add a comment" is a real (if oddly named) placeholder category on the source site itself, 
# not a scraping error — we verified this by checking the raw breadcrumb HTML.
# We'll relabel it as "Uncategorized" for clarity in our analysis.

df.loc[df["Category"] == "Add a comment", "Category"] = "Uncategorized"

# Confirm the fix
df["Category"].value_counts()

Category
Sequential Art        14
Nonfiction            12
Default                9
Poetry                 7
Uncategorized          5
Food and Drink         5
Fiction                5
History                4
Young Adult            4
Fantasy                4
Mystery                3
Music                  3
Thriller               3
Childrens              3
Science Fiction        2
Romance                2
Spirituality           2
Philosophy             2
Contemporary           1
Horror                 1
Health                 1
Science                1
Business               1
New Adult              1
Art                    1
Historical Fiction     1
Travel                 1
Politics               1
Self Help              1
Name: count, dtype: int64

In [15]:
#check a book that has "Default" as its category:
sample_default = df[df["Category"] == "Default"]
print(sample_default[["Title", "Category"]])

                                                Title Category
7   The Coming Woman: A Novel Based on the Life of...  Default
8   The Boys in the Boat: Nine Americans and Their...  Default
10     Starving Hearts (Triangular Trade Trilogy, #1)  Default
26  America's Cradle of Quarterbacks: Western Penn...  Default
27                     Aladdin and His Wonderful Lamp  Default
35                                        Penny Maybe  Default
36     Maude (1883-1993):She Grew Up with the country  Default
65  The Inefficiency Assassin: Time Management Tac...  Default
74                                        Soul Reader  Default


In [17]:
#search for one of these titles on its page and verify:
page_url = "https://books.toscrape.com/catalogue/page-2.html"
response = requests.get(page_url)
response.encoding = 'utf-8'
soup_page = BeautifulSoup(response.text, "html.parser")

books_on_page = soup_page.find_all("article", class_="product_pod")

for book in books_on_page:
    title = book.h3.a["title"]
    if "Aladdin" in title:
        print("Found:", title)
        link = urljoin(page_url, book.h3.a["href"])
        print("Link:", link)

Found: Aladdin and His Wonderful Lamp
Link: https://books.toscrape.com/catalogue/aladdin-and-his-wonderful-lamp_973/index.html


In [19]:
#confirm whether "Default" is another genuine
response3 = requests.get("https://books.toscrape.com/catalogue/aladdin-and-his-wonderful-lamp_973/index.html")
response3.encoding = 'utf-8'
soup_aladdin = BeautifulSoup(response3.text, "html.parser")

breadcrumb_aladdin = soup_aladdin.find("ul", class_="breadcrumb")
print(breadcrumb_aladdin.prettify())

<ul class="breadcrumb">
 <li>
  <a href="../../index.html">
   Home
  </a>
 </li>
 <li>
  <a href="../category/books_1/index.html">
   Books
  </a>
 </li>
 <li>
  <a href="../category/books/default_15/index.html">
   Default
  </a>
 </li>
 <li class="active">
  Aladdin and His Wonderful Lamp
 </li>
</ul>



In [20]:
#relabel this one too:
# "Default" is also a genuine placeholder category on the source site (verified via breadcrumb HTML),
# not a scraping error. Relabeling both known site quirks as "Uncategorized".

df.loc[df["Category"] == "Default", "Category"] = "Uncategorized"

# Confirm
df["Category"].value_counts()

Category
Uncategorized         14
Sequential Art        14
Nonfiction            12
Poetry                 7
Food and Drink         5
Fiction                5
Young Adult            4
History                4
Fantasy                4
Thriller               3
Childrens              3
Music                  3
Mystery                3
Spirituality           2
Science Fiction        2
Romance                2
Philosophy             2
Contemporary           1
Horror                 1
Health                 1
Science                1
Politics               1
New Adult              1
Travel                 1
Art                    1
Business               1
Historical Fiction     1
Self Help              1
Name: count, dtype: int64

In [21]:
#Does price vary by category?
category_price = df.groupby("Category")["Price (INR)"].mean().sort_values(ascending=False)
print(category_price)

Category
Historical Fiction    5642.7000
Politics              5389.6500
Childrens             5168.1000
Health                5150.2500
Self Help             4866.7500
Travel                4742.8500
New Adult             4732.3500
Art                   4638.9000
Fiction               4606.5600
Music                 4535.6500
Science               4510.8000
Mystery               4338.2500
Horror                4121.2500
Philosophy            3887.6250
Science Fiction       3846.1500
Poetry                3805.0500
Food and Drink        3663.0300
History               3563.7000
Business              3500.7000
Nonfiction            3410.6625
Sequential Art        3350.1750
Contemporary          3335.8500
Romance               3139.5000
Thriller              3110.8000
Fantasy               2980.4250
Uncategorized         2950.0500
Young Adult           2629.9875
Spirituality          2619.7500
Name: Price (INR), dtype: float64


In [22]:
#combine average price with sample size, so we don't misread small samples as strong patterns:
category_summary = df.groupby("Category").agg(
    avg_price=("Price (INR)", "mean"),
    count=("Category", "count")
).sort_values("avg_price", ascending=False)

print(category_summary)

                    avg_price  count
Category                            
Historical Fiction  5642.7000      1
Politics            5389.6500      1
Childrens           5168.1000      3
Health              5150.2500      1
Self Help           4866.7500      1
Travel              4742.8500      1
New Adult           4732.3500      1
Art                 4638.9000      1
Fiction             4606.5600      5
Music               4535.6500      3
Science             4510.8000      1
Mystery             4338.2500      3
Horror              4121.2500      1
Philosophy          3887.6250      2
Science Fiction     3846.1500      2
Poetry              3805.0500      7
Food and Drink      3663.0300      5
History             3563.7000      4
Business            3500.7000      1
Nonfiction          3410.6625     12
Sequential Art      3350.1750     14
Contemporary        3335.8500      1
Romance             3139.5000      2
Thriller            3110.8000      3
Fantasy             2980.4250      4
U

In [23]:
#Is there a relationship between price and rating?
df[["Price (INR)", "Rating"]].corr()

,Price (INR),Rating
Price (INR),1.000000,-0.121741
Rating,-0.121741,1.000000


In [24]:
#Does stock count relate to price or rating?
df[["Price (INR)", "Rating", "Stock Count"]].corr()

,Price (INR),Rating,Stock Count
Price (INR),1.000000,-0.121741,0.108314
Rating,-0.121741,1.000000,0.063604
Stock Count,0.108314,0.063604,1.000000


In [25]:
#Are there any outliers or anomalies?
#find the most expensive and cheapest books:
print("Most expensive books:")
print(df.nlargest(3, "Price (INR)")[["Title", "Price (INR)", "Category"]])

print("\nCheapest books:")
print(df.nsmallest(3, "Price (INR)")[["Title", "Price (INR)", "Category"]])

Most expensive books:
                                                Title  Price (INR)    Category
68       The Death of Humanity: and the Case for Life      6101.55  Philosophy
40                     Slow States of Collapse: Poems      6017.55      Poetry
15  Our Band Could Be Your Life: Scenes from the A...      6011.25       Music

Cheapest books:
                                               Title  Price (INR)  \
84                                          Patience       1066.8   
20                                       In Her Wake       1348.2   
81  Princess Between Worlds (Wide-Awake Princess #5)       1400.7   

          Category  
84  Sequential Art  
20        Thriller  
81         Fantasy  


In [27]:
#check for any books with very low stock:
print(df[df["Stock Count"] < 5][["Title", "Stock Count"]])

Empty DataFrame
Columns: [Title, Stock Count]
Index: []
